# Multi-Representation Shallow Encoders for LLMs — minimal run

**Question:** under the same trainable-parameter budget, is it better to use one large encoder for a single 3D representation, or several small encoders (mesh / point cloud / multi-view) aggregated into a frozen LLM?

Pipeline: synthetic 3D scenes → 3 representations → shallow encoders → aggregator (MLP or Q-Former) → K soft tokens → frozen LLM (Qwen2.5-0.5B-Instruct).

Runs on a free Colab T4. Steps: (1) install, (2) config, (3) generate data, (4) models, (5) load LLM, (6) train/eval, (7) run the budget sweep, (8) results.


## 1. Install

In [ ]:
!pip -q install trimesh transformers accelerate

## 2. Config

In [ ]:
import math, json, os, random, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

CFG = dict(
    seed=0,
    n_train=6000, n_test=600,
    n_points=512,          # point cloud size (xyz only, no color)
    view_res=16, n_views=3,  # low-res colored renders (front / side / top)
    v_max=192,             # max mesh vertices per scene (padded)
    K=16,                  # number of soft tokens fed to the LLM (fixed for all settings)
    epochs=20, batch_size=32, lr=5e-4, weight_decay=0.01,
    rep_dropout=0.3,       # drop a whole representation during training (multi-rep only); needs longer training
    llm_name="Qwen/Qwen2.5-0.5B-Instruct",
)
device = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(CFG["seed"])
print("device:", device)

## 3. Synthetic 3D scenes → mesh graph / point cloud (xyz) / multi-view renders (RGB + depth)
Questions are balanced over 4 tasks. Categories are hypotheses; verify them with the single-rep runs.

In [ ]:
# ---------- Synthetic 3D scenes ----------
import trimesh
from trimesh.transformations import random_rotation_matrix

SHAPES = ["cube", "sphere", "cylinder", "cone", "torus"]
COLORS = {"red": (0.9, 0.15, 0.15), "green": (0.15, 0.75, 0.2),
          "blue": (0.15, 0.3, 0.9), "yellow": (0.95, 0.85, 0.1)}
TASKS = {  # task -> category (to be verified empirically with single-rep models)
    "shape": "common",          # "What shape is the highest object?"
    "color": "rep-specific",    # color only exists in the multi-view renders
    "count": "rep-specific",    # mesh connected components (objects may touch)
    "shape_of_color": "combination",  # needs color (views) + fine shape (points/mesh)
}

def make_primitive(shape):
    # deliberately low-resolution meshes (keeps the graph small)
    if shape == "cube":     m = trimesh.creation.box(extents=(1.4, 1.4, 1.4))
    if shape == "sphere":   m = trimesh.creation.icosphere(subdivisions=1, radius=0.85)
    if shape == "cylinder": m = trimesh.creation.cylinder(radius=0.7, height=1.5, sections=10)
    if shape == "cone":     m = trimesh.creation.cone(radius=0.8, height=1.6, sections=10)
    if shape == "torus":    m = trimesh.creation.torus(major_radius=0.65, minor_radius=0.25,
                                                       major_sections=10, minor_sections=6)
    return m

def make_scene(rng):
    n = int(rng.integers(1, 4))
    shapes = [SHAPES[i] for i in rng.integers(0, len(SHAPES), n)]
    colors = list(rng.choice(list(COLORS), n, replace=False))
    zs = np.sort(rng.uniform(-0.9, 0.9, n))
    while n > 1 and np.min(np.diff(zs)) < 0.25:          # distinct heights for "highest"
        zs = np.sort(rng.uniform(-0.9, 0.9, n))
    objs = []
    for s, c, z in zip(shapes, colors, rng.permutation(zs)):
        m = make_primitive(s)
        m.apply_transform(random_rotation_matrix(rng.random(3)))
        m.apply_scale(rng.uniform(0.25, 0.42))
        m.apply_translation([rng.uniform(-0.8, 0.8), rng.uniform(-0.8, 0.8), z])
        objs.append(dict(shape=s, color=c, mesh=m, z=z))
    return objs

def to_mesh_graph(objs, v_max):
    xs, edges, off = [], [], 0
    for o in objs:
        m = o["mesh"]
        xs.append(np.concatenate([m.vertices, m.vertex_normals], 1))
        edges.append(m.edges_unique + off); off += len(m.vertices)
    x = np.concatenate(xs).astype(np.float32); e = np.concatenate(edges).astype(np.int16)
    assert len(x) <= v_max, len(x)
    return x, e

def to_points(objs, n_points, rng):
    m = trimesh.util.concatenate([o["mesh"] for o in objs])
    pts, fid = trimesh.sample.sample_surface(m, n_points, seed=int(rng.integers(1 << 30)))
    owner = np.repeat(np.arange(len(objs)), [len(o["mesh"].faces) for o in objs])[fid]
    cname = np.array([objs[i]["color"] for i in owner])
    perm = rng.permutation(n_points)
    # the encoder only sees xyz; the color names are kept for the textify baseline only
    return pts[perm].astype(np.float32), cname[perm]

def to_views(objs, res, rng, n_dense=4000):
    # orthographic z-buffer splatting: RGB + depth + mask, no external renderer needed
    pts, cols = [], []
    areas = np.array([o["mesh"].area for o in objs]); counts = np.maximum(50, (n_dense * areas / areas.sum()).astype(int))
    for o, k in zip(objs, counts):
        p, _ = trimesh.sample.sample_surface(o["mesh"], int(k), seed=int(rng.integers(1 << 30)))
        pts.append(p); cols.append(np.tile(COLORS[o["color"]], (len(p), 1)))
    P, C = np.concatenate(pts), np.concatenate(cols)
    axes = [(0, 2, 1, -1), (1, 2, 0, +1), (0, 1, 2, +1)]     # (u, v, depth axis, viewing sign): front, side, top
    out = np.zeros((len(axes), 5, res, res), np.float32)
    for vi, (u, v, d, sgn) in enumerate(axes):
        iu = np.clip(((P[:, u] + 1.5) / 3.0 * res).astype(int), 0, res - 1)
        iv = np.clip(((1.5 - P[:, v]) / 3.0 * res).astype(int), 0, res - 1)
        depth = sgn * P[:, d]                                 # larger = closer to the camera
        order = np.argsort(depth)                             # far -> near, near overwrites
        out[vi, 0:3, iv[order], iu[order]] = C[order]          # advanced indexing -> (N, 3)
        out[vi, 3, iv[order], iu[order]] = (depth[order] + 1.5) / 3.0
        out[vi, 4, iv[order], iu[order]] = 1.0
    return out

def make_question(objs, rng, task=None):
    task = task or rng.choice(list(TASKS))
    top = max(objs, key=lambda o: o["z"])
    if task == "shape":
        return task, "What shape is the highest object?", top["shape"]
    if task == "color":
        return task, "What color is the highest object?", top["color"]
    if task == "count":
        return task, "How many objects are there?", str(len(objs))
    o = objs[int(rng.integers(len(objs)))]
    return "shape_of_color", f"What shape is the {o['color']} object?", o["shape"]

def build_split(n, seed):
    rng = np.random.default_rng(seed)
    data = []
    for i in range(n):
        objs = make_scene(rng)
        task, q, a = make_question(objs, rng, task=list(TASKS)[i % len(TASKS)])   # balanced tasks
        mx, me = to_mesh_graph(objs, CFG["v_max"])
        pts, pcol = to_points(objs, CFG["n_points"], rng)
        data.append(dict(points=pts, point_colors=pcol, views=to_views(objs, CFG["view_res"], rng),
                         mesh_x=mx, mesh_e=me, task=task, q=q, a=a,
                         scene=[(o["shape"], o["color"], float(o["z"])) for o in objs]))
    return data

t0 = time.time()
train_data = build_split(CFG["n_train"], seed=1)
test_data = build_split(CFG["n_test"], seed=2)
print(f"generated {len(train_data)}/{len(test_data)} scenes in {time.time()-t0:.0f}s")
print(test_data[0]["q"], "->", test_data[0]["a"])

## 4. Shallow encoders, aggregators (Q-Former / MLP), budget matching
`match_budget` picks the width so that **encoders + aggregator + output projection** hit the target parameter count (LLM excluded). Each representation gets its own LayerNorm so no encoder dominates by scale.

In [ ]:
# ---------- Shallow encoders ----------
class PointEnc(nn.Module):
    """Mini PointNet: shared MLP per point, max-pooled into G local groups -> G tokens."""
    def __init__(self, w, G=32):
        super().__init__(); self.G = G
        self.mlp = nn.Sequential(nn.Linear(3, w), nn.GELU(), nn.Linear(w, w), nn.GELU(), nn.Linear(w, w))
        self.pos = nn.Linear(3, w)
    def forward(self, pts):                                    # (B, N, 3)
        B, N, _ = pts.shape
        centers = pts[:, :self.G]                              # points are pre-shuffled -> random centers
        assign = torch.cdist(pts, centers).argmin(-1)          # (B, N)
        h = self.mlp(pts)
        tok = torch.full((B, self.G, h.size(-1)), -1e4, device=pts.device, dtype=h.dtype)
        tok = tok.scatter_reduce(1, assign[..., None].expand_as(h), h, reduce="amax", include_self=True)
        return tok + self.pos(centers), torch.zeros(B, self.G, dtype=torch.bool, device=pts.device)

class ViewEnc(nn.Module):
    """Small CNN shared across views; each view -> 4x4 spatial tokens."""
    def __init__(self, w, n_views=3):
        super().__init__()
        c = max(8, w // 2)
        self.cnn = nn.Sequential(nn.Conv2d(5, c, 3, padding=1), nn.GELU(),
                                 nn.Conv2d(c, w, 3, stride=2, padding=1), nn.GELU(),
                                 nn.Conv2d(w, w, 3, stride=2, padding=1))
        self.pos = nn.Parameter(torch.randn(n_views * 16, w) * 0.02)
    def forward(self, views):                                  # (B, V, 5, H, W)
        B, V = views.shape[:2]
        h = self.cnn(views.flatten(0, 1))                      # (B*V, w, 4, 4)
        tok = h.flatten(2).transpose(1, 2).reshape(B, V * 16, -1) + self.pos
        return tok, torch.zeros(B, tok.size(1), dtype=torch.bool, device=views.device)

class MeshEnc(nn.Module):
    """Dense GIN on the mesh vertex graph -> one token per vertex."""
    def __init__(self, w, layers=2):
        super().__init__()
        self.inp = nn.Linear(6, w)
        self.eps = nn.Parameter(torch.zeros(layers))
        self.mlps = nn.ModuleList([nn.Sequential(nn.Linear(w, w), nn.GELU(), nn.Linear(w, w)) for _ in range(layers)])
    def forward(self, x, adj, pad):                            # (B,V,6), (B,V,V), (B,V) True=pad
        h = self.inp(x)
        for e, mlp in zip(self.eps, self.mlps):
            h = h + mlp((1 + e) * h + adj @ h)
            h = h.masked_fill(pad[..., None], 0)
        return h, pad

ENCODERS = {"point": PointEnc, "view": ViewEnc, "mesh": MeshEnc}

# ---------- Aggregators ----------
class QFormerAgg(nn.Module):
    def __init__(self, d, K, layers=1, heads=4):
        super().__init__()
        self.q = nn.Parameter(torch.randn(K, d) * 0.02)
        self.layers = nn.ModuleList([nn.TransformerDecoderLayer(d, heads, 2 * d, dropout=0.0, batch_first=True, norm_first=True)
                                     for _ in range(layers)])
        self.norm = nn.LayerNorm(d)
    def forward(self, mem, pad):
        z = self.q.expand(mem.size(0), -1, -1)
        for l in self.layers: z = l(z, mem, memory_key_padding_mask=pad)
        return self.norm(z)

class MLPAgg(nn.Module):
    """Per-representation masked mean+max pooling -> MLP -> K tokens."""
    def __init__(self, d, K, n_reps):
        super().__init__(); self.K, self.d = K, d
        self.mlp = nn.Sequential(nn.Linear(2 * d * n_reps, 2 * d), nn.GELU(), nn.Linear(2 * d, K * d))
    def forward(self, toks, pads):                             # lists per representation
        feats = []
        for t, p in zip(toks, pads):
            keep = (~p).float()[..., None]
            mean = (t * keep).sum(1) / keep.sum(1).clamp(min=1)
            mx = t.masked_fill(p[..., None], -1e4).max(1).values * (keep.sum(1) > 0)
            feats += [mean, mx]
        return self.mlp(torch.cat(feats, -1)).view(-1, self.K, self.d)

class MultiRepAdaptor(nn.Module):
    def __init__(self, reps, w, agg="qformer", K=16, llm_dim=896, agg_layers=1):
        super().__init__()
        self.reps, self.agg_type = reps, agg
        self.enc = nn.ModuleDict({r: ENCODERS[r](w) for r in reps})
        self.norm = nn.ModuleDict({r: nn.LayerNorm(w) for r in reps})   # put every representation on the same scale
        self.type_emb = nn.Parameter(torch.randn(len(reps), w) * 0.02)
        self.agg = QFormerAgg(w, K, agg_layers) if agg == "qformer" else MLPAgg(w, K, len(reps))
        self.out = nn.Linear(w, llm_dim)
    def forward(self, batch, drop=None):                       # drop: (B, n_reps) bool, True = representation removed
        toks, pads = [], []
        for i, r in enumerate(self.reps):
            if r == "point": t, p = self.enc[r](batch["points"])
            if r == "view":  t, p = self.enc[r](batch["views"])
            if r == "mesh":  t, p = self.enc[r](batch["mesh_x"], batch["adj"], batch["mesh_pad"])
            if drop is not None: p = p | drop[:, i:i + 1]
            toks.append(self.norm[r](t) + self.type_emb[i]); pads.append(p)
        if self.agg_type == "qformer":
            mem, pad = torch.cat(toks, 1), torch.cat(pads, 1)
            z = self.agg(mem.masked_fill(pad[..., None], 0), pad)
        else:
            z = self.agg(toks, pads)
        return self.out(z)                                     # (B, K, llm_dim)

def n_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

def match_budget(reps, agg, target, llm_dim=896, K=16):
    """Pick the encoder/aggregator width whose trainable-parameter count is closest to `target`."""
    best = None
    for w in range(16, 1025, 8):
        n = n_params(MultiRepAdaptor(reps, w, agg, K, llm_dim))
        if best is None or abs(n - target) < abs(best[1] - target): best = (w, n)
        if n > target * 1.5: break
    return best   # (width, n_params)

## 5. Frozen LLM

In [ ]:
# ---------- Frozen LLM ----------
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(CFG["llm_name"])
llm = AutoModelForCausalLM.from_pretrained(CFG["llm_name"], torch_dtype=torch.float32).to(device)
llm.requires_grad_(False); llm.eval()
print("LLM hidden size:", llm.config.hidden_size)

## 6. Batching, training, evaluation
Soft tokens go in through `inputs_embeds`; loss only on answer tokens. Do **not** wrap the LLM forward in `torch.no_grad()` during training — gradients must flow through the frozen LLM into the adaptor.

In [ ]:
# ---------- Batching ----------
LLM_DIM = llm.config.hidden_size
embed = llm.get_input_embeddings()
EOS = tok.eos_token_id

def encode_text(ex):
    p = tok(f"Question: {ex['q']} Answer:", add_special_tokens=False)["input_ids"]
    a = tok(" " + ex["a"], add_special_tokens=False)["input_ids"] + [EOS]
    return p, a

def collate(exs):
    B, V = len(exs), CFG["v_max"]
    mesh_x = torch.zeros(B, V, 6); adj = torch.zeros(B, V, V); mesh_pad = torch.ones(B, V, dtype=torch.bool)
    for i, ex in enumerate(exs):
        n = len(ex["mesh_x"]); mesh_x[i, :n] = torch.from_numpy(ex["mesh_x"]); mesh_pad[i, :n] = False
        e = torch.from_numpy(ex["mesh_e"].astype(np.int64)); adj[i, e[:, 0], e[:, 1]] = 1; adj[i, e[:, 1], e[:, 0]] = 1
    texts = [encode_text(ex) for ex in exs]
    L = max(len(p) + len(a) for p, a in texts)
    ids = torch.full((B, L), EOS, dtype=torch.long); labels = torch.full((B, L), -100, dtype=torch.long)
    attn = torch.zeros(B, L, dtype=torch.long)
    for i, (p, a) in enumerate(texts):
        ids[i, :len(p) + len(a)] = torch.tensor(p + a); labels[i, len(p):len(p) + len(a)] = torch.tensor(a)
        attn[i, :len(p) + len(a)] = 1
    return dict(points=torch.from_numpy(np.stack([ex["points"] for ex in exs])),
                views=torch.from_numpy(np.stack([ex["views"] for ex in exs])),
                mesh_x=mesh_x, adj=adj, mesh_pad=mesh_pad, ids=ids, labels=labels, attn=attn,
                task=[ex["task"] for ex in exs])

def to_dev(b): return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}

def forward_llm(model, b, drop=None):
    soft = model(b, drop)                                      # (B, K, H)
    x = torch.cat([soft, embed(b["ids"])], 1)
    attn = torch.cat([torch.ones(soft.shape[:2], dtype=b["attn"].dtype, device=device), b["attn"]], 1)
    labels = torch.cat([torch.full(soft.shape[:2], -100, dtype=torch.long, device=device), b["labels"]], 1)
    return llm(inputs_embeds=x, attention_mask=attn, labels=labels)

# ---------- Train / eval ----------
def train(model, data, cfg=CFG, log_every=50):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    steps = cfg["epochs"] * math.ceil(len(data) / cfg["batch_size"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, cfg["lr"], total_steps=steps, pct_start=0.05)
    model.train(); step = 0
    for ep in range(cfg["epochs"]):
        idx = np.random.permutation(len(data))
        for s in range(0, len(data), cfg["batch_size"]):
            b = to_dev(collate([data[i] for i in idx[s:s + cfg["batch_size"]]]))
            drop = None
            if len(model.reps) > 1 and cfg["rep_dropout"] > 0:
                drop = torch.rand(len(b["ids"]), len(model.reps), device=device) < cfg["rep_dropout"]
                drop[drop.all(1), 0] = False                    # always keep at least one representation
            loss = forward_llm(model, b, drop).loss
            opt.zero_grad(); loss.backward(); opt.step(); sched.step(); step += 1
            if step % log_every == 0: print(f"  ep {ep} step {step}/{steps} loss {loss.item():.3f}")
    return model

@torch.no_grad()
def evaluate(model, data, drop_reps=()):
    """Exact match via teacher forcing: correct iff every answer token (incl. EOS) is the argmax,
    which is exactly what greedy decoding would output."""
    model.eval(); hits = {}
    for s in range(0, len(data), 64):
        b = to_dev(collate(data[s:s + 64]))
        drop = None
        if drop_reps:
            drop = torch.zeros(len(b["ids"]), len(model.reps), dtype=torch.bool, device=device)
            for r in drop_reps: drop[:, model.reps.index(r)] = True
        out = forward_llm(model, b, drop)
        logits = out.logits[:, CFG["K"]:-1]; tgt = b["labels"][:, 1:]
        ok = ((logits.argmax(-1) == tgt) | (tgt == -100)).all(1)
        for t, o in zip(b["task"], ok.tolist()): hits.setdefault(t, []).append(o)
    model.train()
    res = {t: float(np.mean(v)) for t, v in hits.items()}
    res["avg"] = float(np.mean(list(res.values())))
    return res

## 7. Budget sweep
Results are appended to `results.json` after every run, so the cell can be re-run after a Colab disconnect.

In [ ]:
# ---------- Budget sweep ----------
BUDGETS = [0.5e6, 1e6, 2e6]                 # trainable params (encoders + aggregator), LLM excluded
SETTINGS = {                                 # name -> (representations, aggregator)
    "point only": (["point"], "qformer"),
    "view only":  (["view"], "qformer"),
    "mesh only":  (["mesh"], "qformer"),
    "ours (all, Q-Former)": (["mesh", "point", "view"], "qformer"),
    "ours (all, MLP)":      (["mesh", "point", "view"], "mlp"),
}
RESULTS_PATH = "results.json"
results = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else []   # resume after a Colab disconnect
done = {(r["setting"], r["budget"]) for r in results}

for budget in BUDGETS:
    for name, (reps, agg) in SETTINGS.items():
        if (name, budget) in done: continue
        set_seed(CFG["seed"])
        w, n = match_budget(reps, agg, budget, LLM_DIM, CFG["K"])
        model = MultiRepAdaptor(reps, w, agg, CFG["K"], LLM_DIM).to(device)
        print(f"\n== {name} | budget {budget/1e6:.1f}M -> width {w}, {n/1e6:.2f}M params")
        t0 = time.time(); train(model, train_data)
        res = evaluate(model, test_data)
        print({k: round(v, 3) for k, v in res.items()}, f"({time.time()-t0:.0f}s)")
        results.append(dict(setting=name, budget=budget, width=w, params=n, **res))
        json.dump(results, open(RESULTS_PATH, "w"), indent=1)

## 8. Results

In [ ]:
# ---------- Results ----------
import pandas as pd, matplotlib.pyplot as plt
df = pd.DataFrame(results)
cols = ["setting", "budget", "params"] + list(TASKS) + ["avg"]
display(df[cols].round(3))

fig, axes = plt.subplots(1, len(TASKS), figsize=(4 * len(TASKS), 3.2), sharey=True)
for ax, t in zip(axes, TASKS):
    for name, g in df.groupby("setting"):
        g = g.sort_values("params"); ax.plot(g["params"], g[t], marker="o", label=name)
    ax.set_xscale("log"); ax.set_title(f"{t} ({TASKS[t]})"); ax.set_xlabel("trainable params")
axes[0].set_ylabel("exact-match accuracy"); axes[-1].legend(fontsize=7)
plt.tight_layout(); plt.show()

## 9. (Optional) Textify baseline

In [ ]:
# ---------- (Optional) Textify baseline: colored point cloud as text, zero-shot, same LLM ----------
def textify(ex, n=64):
    idx = np.linspace(0, len(ex["points"]) - 1, n).astype(int)        # points are already shuffled
    pts = " ".join(f"({x:.2f},{y:.2f},{z:.2f}) {c}" for (x, y, z), c in zip(ex["points"][idx], ex["point_colors"][idx]))
    return f"Colored 3D point cloud (x,y,z) color: {pts}\nQuestion: {ex['q']} Answer with one word."

@torch.no_grad()
def textify_zero_shot(data, n=100):
    hits = {}
    for ex in data[:n]:
        msgs = [{"role": "user", "content": textify(ex)}]
        ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(device)
        out = llm.generate(ids, max_new_tokens=6, do_sample=False)
        pred = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip().lower().rstrip(".")
        hits.setdefault(ex["task"], []).append(pred == ex["a"])
    res = {t: float(np.mean(v)) for t, v in hits.items()}; res["avg"] = float(np.mean(list(res.values())))
    return res

print("textify zero-shot:", textify_zero_shot(test_data))
print("tokens per textified prompt ≈", len(tok(textify(test_data[0]))["input_ids"]), "vs K =", CFG["K"])

## Notes from the CPU sanity check (no LLM; linear probe on adaptor outputs, 3k train scenes)

| setting | shape | color | count | shape_of_color |
|---|---|---|---|---|
| view only | 0.27 | 0.97 | 0.94 | 0.25 |
| mesh only | 0.87 | 0.25 | 0.74 | 0.63 |
| point only | 0.21 | 0.26 | 0.75 | 0.28 |
| all, Q-Former, 12 epochs, no LayerNorm | 0.89 | 0.23 | 0.87 | 0.65 |
| all, Q-Former, 24 epochs, LayerNorm, rep-dropout 0.3 | 0.93 | 0.89 | 0.87 | 0.67 |

- Representations do carry different information (views → color, mesh → shape), as designed.
- Multi-rep models converge more slowly, and representation dropout slows them further → train to convergence, plot learning curves, don't compare at a fixed small epoch count.
- **Open issue:** `shape_of_color` stays ~0.65 for every setting. Color (views) and shape (mesh) both arrive, but nothing tells the aggregator which view pixel and which mesh vertex belong to the same object. Candidate fixes: shared 3D positional encoding for all tokens (unproject view pixels to 3D), question-conditioned queries.
- The point encoder is weak on shape (near chance) — worth tuning (more groups, local coordinates relative to group centers).
